# Lesson 5.4 — Secrets, Keys, and Identity Management

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Store and load API keys securely via environment/secret managers
- Avoid committing secrets to source control
- Apply authentication, authorization, and access monitoring
- Rotate keys and scope credentials with least privilege

**You will produce:** A secure key-loading helper and a secrets-hygiene checklist/validator.

## Concepts

Credential management spans dev and production:

- **Never hard-code keys**. Load `ANTHROPIC_API_KEY` from environment variables or a secret manager (AWS Secrets Manager, Vault, etc.), not source code.
- **Keep secrets out of git**: use `.gitignore`, `.env` (untracked), and pre-commit secret scanning.
- **AuthN/AuthZ**: validate identity (who) and authorize actions (what they may do); verify access level before privileged operations and monitor authorized access.
- **Rotation & scoping**: rotate keys regularly, scope them with least privilege, and use separate keys per environment so a leak is contained.

> **Reference:** [Anthropic - API keys & security](https://docs.anthropic.com/en/api/getting-started)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Load keys safely

Read from env; fail with a clear message, never a default literal key.

In [ ]:
import os

def load_key(var="ANTHROPIC_API_KEY"):
    key = os.environ.get(var)
    if not key:
        raise RuntimeError(f"{var} not set. Export it or use a secret manager.")
    return key

try:
    load_key()
    print("key loaded from env")
except RuntimeError as e:
    print("guarded:", e)

## Code-Along: Step 2 — Scan for accidentally committed secrets

A simple detector you could run in a pre-commit hook.

In [ ]:
import re
SECRET_PAT = re.compile(r"(sk-[A-Za-z0-9]{8,}|AKIA[0-9A-Z]{16})")

def scan_for_secrets(text):
    return SECRET_PAT.findall(text)

sample_diff = 'client = Anthropic(api_key="sk-ABCD12345678")'
print("found secrets:", scan_for_secrets(sample_diff))

## Exercise

Write `secrets_hygiene_report(config)` that returns a list of violations: hard-coded key, missing `.gitignore` entry for `.env`, and same key reused across environments.

In [ ]:
def secrets_hygiene_report(config):
    # TODO: check config['hardcoded_key'], config['gitignore_has_env'], config['per_env_keys']
    # TODO: return a list of human-readable violation strings
    pass

## Key Takeaways

- Load keys from env/secret managers; never hard-code or commit them.
- Use .gitignore and secret scanning to keep credentials out of source control.
- Validate identity (authN) and authorize actions (authZ) before privileged operations.
- Rotate and scope keys with least privilege; separate keys per environment.

---

**Next:** [Lesson 5.5 — Debugging, Error Handling, and Trace Analysis](./lesson-5.5-debugging-and-trace-analysis.ipynb)